# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from catboost import CatBoostClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')

y_train = pd.read_parquet('../data/y_train.parquet')

In [5]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


In [6]:
X_test.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.070991
439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,...,0.298107,0.185508,0.178575,0.147360,0.298107,0.298107,0.321361,0.178575,0.147360,0.253721
439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,...,0.298107,0.207392,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.070991
439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,...,0.102471,0.207392,0.178575,0.279021,0.102471,0.102471,0.071042,0.178575,0.279021,0.216120
439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,...,0.298107,0.207392,0.178575,0.111582,0.298107,0.298107,0.206543,0.178575,0.111582,0.216120


## Creating Stacking Database

In [7]:
lgbm = load_pickle('../models/model_lightgbm.pkl')
cat = load_pickle('../models/model_pure_catboost.pkl')
catencoder_cat = load_pickle('../models/model_catboostencoder_catboost.pkl')
xgb = load_pickle('../models/model_xgboost.pkl')
hist = load_pickle('../models/model_histgradientboosting.pkl')
lg = load_pickle('../models/model_logistic_regression.pkl')
voting = load_pickle('../models/model_voting.pkl')

In [ ]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

lgbm_proba = cross_val_predict(lgbm, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
catencoder_cat_proba = cross_val_predict(catencoder_cat, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
xgb_proba = cross_val_predict(xgb, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
hist_proba = cross_val_predict(hist, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
lg_proba = cross_val_predict(lg, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
voting_proba = cross_val_predict(voting, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba',)[:, 1]

cat_proba = cross_val_predict(
    make_pipeline(
        DropFeatures(['treelaptime_s', 'treeraceprogress_position', 'treeposition_laptime_s']),
        CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='AUC',
            auto_class_weights="Balanced",
            verbose=0,
            thread_count=1
        )
    ), 
    X_train, 
    y_train.PitNextLap,
    cv=cv, 
    n_jobs=-1, 
    method='predict_proba',
    params={'catboostclassifier__cat_features': ['driver', 'compound', 'race']}
)[:, 1]

In [ ]:
X_train_stacking = pd.DataFrame({
    'lgbm_proba': lgbm_proba,
    'catencoder_cat_proba': catencoder_cat_proba,
    'xgb_proba': xgb_proba,
    'hist_proba': hist_proba,
    'lg_proba': lg_proba,
    'voting_proba': voting_proba,
    'cat_proba': cat_proba
})

X_train_stacking.to_parquet('../data/X_train_stacking.parquet')

In [ ]:
X_test_stacking = pd.DataFrame({
    'lgbm_proba': lgbm.predict_proba(X_test)[:, 1],
    'catencoder_cat_proba': catencoder_cat.predict_proba(X_test)[:, 1],
    'xgb_proba': xgb.predict_proba(X_test)[:, 1],
    'hist_proba': hist.predict_proba(X_test)[:, 1],
    'lg_proba': lg.predict_proba(X_test)[:, 1],
    'voting_proba': voting.predict_proba(X_test)[:, 1],
    'cat_proba': cat.predict_proba(X_test)[:, 1]
})

X_test_stacking.to_parquet('../data/X_test_stacking.parquet')

# Machine Learning

In [52]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000),
        "random_state": trial.suggest_int("random_state", 0, 1000)
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_stacking, y_train), n_trials=500, n_jobs=-1)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-13 12:12:35,144] A new study created in memory with name: no-name-76bf0235-cb5b-4921-a539-94e80a1f6edb
[I 2026-05-13 12:12:38,952] Trial 9 finished with value: 0.5 and parameters: {'solver': 'saga', 'C': 1.6473915438563915e-05, 'l1_ratio': 0.9122264251617757, 'class_weight': None, 'fit_intercept': False, 'tol': 2.5810005378995377e-05, 'max_iter': 4153, 'random_state': 334}. Best is trial 9 with value: 0.5.
[I 2026-05-13 12:12:41,191] Trial 7 finished with value: 0.5 and parameters: {'solver': 'saga', 'C': 1.1823904139199353e-05, 'l1_ratio': 0.8238443705178026, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.0017001359671289836, 'max_iter': 1418, 'random_state': 400}. Best is trial 9 with value: 0.5.
[I 2026-05-13 12:12:45,542] Trial 1 finished with value: 0.7881533219107638 and parameters: {'solver': 'saga', 'C': 0.11744159885812057, 'l1_ratio': 0.9401668857519256, 'class_weight': None, 'fit_intercept': False, 'tol': 0.003412336600702748, 'max_iter': 4596, 'rando

Best trial score:
0.9498706450715897

Best params:
{'solver': 'saga', 'C': 0.01185232258736047, 'l1_ratio': 0.9726417663956377, 'class_weight': None, 'fit_intercept': True, 'tol': 1.6007186630931186e-05, 'max_iter': 1218, 'random_state': 290}


In [55]:
stacking_lg = LogisticRegression(**study.best_params).fit(X_stacking, y_train.PitNextLap)

# Submission

In [56]:
submission = pd.read_csv('../data/sample_submission.csv')

In [57]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test_stacking)[:, 1]

In [58]:
submission.to_csv('../data/submission.csv', index=False)